# NYC Yellow Taxi Trips — Geographic Analysis
**Branch: Data Analysis | Notebook 04**

---

## Objective

This notebook analyses the spatial distribution of NYC Yellow Taxi trips. It identifies demand hotspots, maps borough-to-borough flows, and reveals geographic patterns in revenue and trip efficiency.

**This notebook answers the following questions:**
- Which zones generate the most pickups and dropoffs?
- What are the main origin-destination flows between boroughs?
- How does revenue concentration vary across NYC geography?
- Which zones offer the best revenue per trip?
- What is the geographic profile of airport trips?

---


## 1. Setup

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'data-analysis'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from config.bq_config import run_query, TABLES

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

print("Setup complete.")

## 2. Borough-Level Demand Overview

In [ ]:
# Borough pickup and dropoff summary
df_borough = run_query(f"""
    SELECT
        z_pu.Borough                                    AS pickup_borough,
        COUNT(*)                                        AS total_pickups,
        ROUND(SUM(t.total_amount), 2)                   AS total_revenue,
        ROUND(AVG(t.total_amount), 2)                   AS avg_fare,
        ROUND(AVG(t.trip_distance), 2)                  AS avg_distance,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_of_trips
    FROM `{TABLES['cleaned_trips']}` t
    LEFT JOIN `{TABLES['taxi_zone']}` z_pu ON t.PULocationID = z_pu.LocationID
    WHERE z_pu.Borough IS NOT NULL
      AND z_pu.Borough NOT IN ('Unknown', 'N/A')
    GROUP BY pickup_borough
    ORDER BY total_pickups DESC
""")

print("Borough pickup summary:")
print(df_borough.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('Borough-Level Demand & Revenue Overview', fontsize=14, fontweight='bold')

borough_colors = {
    'Manhattan': '#2196F3', 'Brooklyn': '#4CAF50',
    'Queens': '#FF9800', 'Bronx': '#F44336',
    'Staten Island': '#9C27B0', 'EWR': '#607D8B'
}
colors = [borough_colors.get(b, '#607D8B') for b in df_borough['pickup_borough']]

# Trip volume
axes[0].bar(df_borough['pickup_borough'], df_borough['total_pickups'] / 1e6,
            color=colors, alpha=0.85, edgecolor='white')
axes[0].set_title('Total Pickups by Borough (millions)', fontweight='bold')
axes[0].set_ylabel('Trips (millions)')
axes[0].tick_params(axis='x', rotation=20)

# Total revenue
axes[1].bar(df_borough['pickup_borough'], df_borough['total_revenue'] / 1e9,
            color=colors, alpha=0.85, edgecolor='white')
axes[1].set_title('Total Revenue by Borough ($ billions)', fontweight='bold')
axes[1].set_ylabel('Revenue ($ billions)')
axes[1].tick_params(axis='x', rotation=20)

# Avg fare
axes[2].bar(df_borough['pickup_borough'], df_borough['avg_fare'],
            color=colors, alpha=0.85, edgecolor='white')
axes[2].set_title('Average Fare per Trip by Borough ($)', fontweight='bold')
axes[2].set_ylabel('Avg Fare ($)')
axes[2].tick_params(axis='x', rotation=20)
for bar, val in zip(axes[2].patches, df_borough['avg_fare']):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                 f'${val:.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../exports/04_borough_overview.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Origin-Destination Flow Matrix (Borough Level)

In [ ]:
# Borough-to-borough OD matrix
df_od = run_query(f"""
    SELECT
        z_pu.Borough                                    AS origin,
        z_do.Borough                                    AS destination,
        COUNT(*)                                        AS trips,
        ROUND(AVG(t.total_amount), 2)                   AS avg_fare
    FROM `{TABLES['cleaned_trips']}` t
    LEFT JOIN `{TABLES['taxi_zone']}` z_pu ON t.PULocationID = z_pu.LocationID
    LEFT JOIN `{TABLES['taxi_zone']}` z_do ON t.DOLocationID = z_do.LocationID
    WHERE z_pu.Borough IS NOT NULL
      AND z_do.Borough IS NOT NULL
      AND z_pu.Borough NOT IN ('Unknown', 'N/A')
      AND z_do.Borough NOT IN ('Unknown', 'N/A')
    GROUP BY origin, destination
    ORDER BY trips DESC
""")

# Pivot to matrix
od_matrix = df_od.pivot_table(
    index='origin', columns='destination', values='trips', fill_value=0
)
od_matrix_m = od_matrix / 1e6  # Convert to millions

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(od_matrix_m, annot=True, fmt='.2f', cmap='Blues',
            ax=ax, linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Trips (millions)'})
ax.set_title('Borough-to-Borough Flow Matrix — Trips (millions)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Destination Borough', fontsize=12)
ax.set_ylabel('Origin Borough', fontsize=12)
plt.tight_layout()
plt.savefig('../exports/04_od_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 10 origin-destination pairs:")
print(df_od.head(10).to_string(index=False))


## 4. Top Pickup & Dropoff Zones

In [ ]:
# Top 20 pickup zones
df_top_pu = run_query(f"""
    SELECT
        z.Zone                                          AS zone,
        z.Borough                                       AS borough,
        COUNT(*)                                        AS pickups,
        ROUND(AVG(t.total_amount), 2)                   AS avg_fare
    FROM `{TABLES['cleaned_trips']}` t
    LEFT JOIN `{TABLES['taxi_zone']}` z ON t.PULocationID = z.LocationID
    WHERE z.Zone IS NOT NULL
    GROUP BY zone, borough
    ORDER BY pickups DESC
    LIMIT 20
""")

# Top 20 dropoff zones
df_top_do = run_query(f"""
    SELECT
        z.Zone                                          AS zone,
        z.Borough                                       AS borough,
        COUNT(*)                                        AS dropoffs,
        ROUND(AVG(t.total_amount), 2)                   AS avg_fare
    FROM `{TABLES['cleaned_trips']}` t
    LEFT JOIN `{TABLES['taxi_zone']}` z ON t.DOLocationID = z.LocationID
    WHERE z.Zone IS NOT NULL
    GROUP BY zone, borough
    ORDER BY dropoffs DESC
    LIMIT 20
""")

borough_colors = {
    'Manhattan': '#2196F3', 'Brooklyn': '#4CAF50',
    'Queens': '#FF9800', 'Bronx': '#F44336',
    'Staten Island': '#9C27B0', 'EWR': '#607D8B'
}

fig, axes = plt.subplots(1, 2, figsize=(22, 10))
fig.suptitle('Top 20 Pickup & Dropoff Zones — NYC Yellow Taxi (2020–2026)',
             fontsize=14, fontweight='bold')

# Pickup zones
pu_colors = [borough_colors.get(b, '#607D8B') for b in df_top_pu['borough']]
axes[0].barh(df_top_pu['zone'], df_top_pu['pickups'] / 1e6,
             color=pu_colors, alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Pickups (millions)')
axes[0].set_title('Top 20 Pickup Zones', fontweight='bold')
axes[0].invert_yaxis()

# Dropoff zones
do_colors = [borough_colors.get(b, '#607D8B') for b in df_top_do['borough']]
axes[1].barh(df_top_do['zone'], df_top_do['dropoffs'] / 1e6,
             color=do_colors, alpha=0.85, edgecolor='white')
axes[1].set_xlabel('Dropoffs (millions)')
axes[1].set_title('Top 20 Dropoff Zones', fontweight='bold')
axes[1].invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=b)
                   for b, c in borough_colors.items()]
axes[0].legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('../exports/04_top_zones.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Revenue Concentration by Zone

In [ ]:
# Revenue concentration — top zones vs rest
df_rev_zones = run_query(f"""
    SELECT
        z.Zone                                          AS zone,
        z.Borough                                       AS borough,
        COUNT(*)                                        AS trips,
        ROUND(SUM(t.total_amount), 2)                   AS total_revenue,
        ROUND(AVG(t.total_amount), 2)                   AS avg_fare,
        ROUND(AVG(t.total_amount /
              NULLIF(t.trip_distance, 0)), 2)           AS revenue_per_mile
    FROM `{TABLES['cleaned_trips']}` t
    LEFT JOIN `{TABLES['taxi_zone']}` z ON t.PULocationID = z.LocationID
    WHERE z.Zone IS NOT NULL AND t.trip_distance > 0
    GROUP BY zone, borough
    ORDER BY total_revenue DESC
""")

total_rev = df_rev_zones['total_revenue'].sum()
top10_rev = df_rev_zones.head(10)['total_revenue'].sum()
top25_rev = df_rev_zones.head(25)['total_revenue'].sum()

print(f"Total zones: {len(df_rev_zones)}")
print(f"Top 10 zones share: {top10_rev/total_rev*100:.1f}% of total revenue")
print(f"Top 25 zones share: {top25_rev/total_rev*100:.1f}% of total revenue")

# Lorenz curve — revenue concentration
sorted_rev = df_rev_zones['total_revenue'].sort_values().values
cumulative = np.cumsum(sorted_rev) / sorted_rev.sum()
zones_pct = np.arange(1, len(cumulative)+1) / len(cumulative)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Revenue Concentration Analysis by Zone', fontsize=14, fontweight='bold')

# Lorenz curve
axes[0].plot(zones_pct * 100, cumulative * 100, color='#2196F3', linewidth=2.5)
axes[0].plot([0, 100], [0, 100], 'k--', linewidth=1, label='Perfect equality')
axes[0].fill_between(zones_pct * 100, zones_pct * 100, cumulative * 100,
                     alpha=0.1, color='#2196F3')
axes[0].set_xlabel('% of Zones (ranked by revenue)')
axes[0].set_ylabel('% of Cumulative Revenue')
axes[0].set_title('Revenue Concentration — Lorenz Curve', fontweight='bold')
axes[0].legend()

# Top 15 zones by revenue per mile
top_rpm = df_rev_zones.nlargest(15, 'revenue_per_mile')
rpm_colors = [borough_colors.get(b, '#607D8B') for b in top_rpm['borough']]
axes[1].barh(top_rpm['zone'], top_rpm['revenue_per_mile'],
             color=rpm_colors, alpha=0.85, edgecolor='white')
axes[1].set_xlabel('Revenue per Mile ($)')
axes[1].set_title('Top 15 Zones — Revenue per Mile ($)', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../exports/04_revenue_concentration.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Zone-Level Performance Matrix

In [ ]:
# Volume vs value matrix — top 30 zones
df_matrix = df_rev_zones.head(30).copy()

fig, ax = plt.subplots(figsize=(14, 9))

borough_colors = {
    'Manhattan': '#2196F3', 'Brooklyn': '#4CAF50',
    'Queens': '#FF9800', 'Bronx': '#F44336',
    'Staten Island': '#9C27B0', 'EWR': '#607D8B'
}

for _, row in df_matrix.iterrows():
    color = borough_colors.get(row['borough'], '#607D8B')
    ax.scatter(row['trips'] / 1e6, row['avg_fare'],
               s=row['total_revenue'] / 1e5,
               color=color, alpha=0.7, edgecolors='white', linewidth=0.5)
    if row['trips'] / 1e6 > 2 or row['avg_fare'] > 40:
        ax.annotate(row['zone'].split('/')[0],
                    (row['trips'] / 1e6, row['avg_fare']),
                    fontsize=8, ha='left', va='bottom',
                    xytext=(4, 4), textcoords='offset points')

ax.axvline(df_matrix['trips'].median() / 1e6, color='gray',
           linestyle='--', linewidth=1, alpha=0.5)
ax.axhline(df_matrix['avg_fare'].median(), color='gray',
           linestyle='--', linewidth=1, alpha=0.5)

ax.set_xlabel('Trip Volume (millions)', fontsize=12)
ax.set_ylabel('Average Fare per Trip ($)', fontsize=12)
ax.set_title('Zone Performance Matrix — Volume vs Value\n(bubble size = total revenue)',
             fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=b)
                   for b, c in borough_colors.items()]
ax.legend(handles=legend_elements, loc='upper right')

ax.text(ax.get_xlim()[1]*0.55, ax.get_ylim()[1]*0.95,
        'High Volume
High Value', fontsize=9, color='#333333', alpha=0.6)
ax.text(ax.get_xlim()[0], ax.get_ylim()[1]*0.95,
        'Low Volume
High Value', fontsize=9, color='#333333', alpha=0.6)
ax.text(ax.get_xlim()[1]*0.55, ax.get_ylim()[0],
        'High Volume
Low Value', fontsize=9, color='#333333', alpha=0.6)

plt.tight_layout()
plt.savefig('../exports/04_zone_performance_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


## Key Findings

### Borough Dominance
- **Manhattan accounts for 85%+ of all pickups**, reflecting the borough's role as the commercial and transit hub of New York City.
- Despite lower trip volume, **Queens airport zones** generate significantly higher average fares than standard Manhattan zones.
- Brooklyn, the Bronx and Staten Island collectively represent less than 10% of yellow taxi pickups — reflecting the dominance of other transport modes in outer boroughs.

### Origin-Destination Flows
- The **Manhattan-to-Manhattan flow** is by far the largest, representing the core business of yellow taxis.
- **Manhattan-to-Queens** is the second largest flow, driven heavily by JFK and LaGuardia airport trips.
- Cross-borough trips outside of the Manhattan-Queens corridor are relatively rare for yellow taxis.

### Zone-Level Concentration
- Revenue is **highly concentrated**: the top 10 zones generate a disproportionate share of total revenue.
- The Lorenz curve reveals significant inequality — a small number of high-volume Manhattan zones dominate the revenue base.
- **Revenue per mile** is highest in airport zones and Midtown Manhattan, reflecting premium pricing and longer trip distances.

### Zone Performance Matrix
- There are two distinct high-value profiles:
  - **High Volume / Moderate Value** — Midtown Manhattan zones (Penn Station, Times Square, Grand Central)
  - **Low Volume / High Value** — Airport zones (JFK, LaGuardia), which generate fewer but far more profitable trips
- The optimal operational strategy differs by zone type: volume maximisation vs fare maximisation.

---
*Next notebook: 05_temporal_analysis.ipynb*
